# Libraries and Load Data

First, we import the necessary libraries for data analysis and visualization. These libraries include:
- pandas for data manipulation;
- numpy for numerical operations;
- matplotlib and seaborn for plotting;
- plotly for interactive visualizations.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")  # Set the theme for seaborn plots

# Enable inline plotting in Jupyter notebooks
%matplotlib inline  

Then we can check the shape of the dataset and display the first few rows to get an overview of the initial data, before the data cleaning.


In [2]:
# load the dataset
df = pd.read_csv('../data/train.csv')

print(f'original shape of the dataset: {df.shape}')
df.head()

original shape of the dataset: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Handle Missing Values

## Age

Now we can check for missing values in the dataset and fill them with appropriate values. For example, we can fill missing values in the 'Age' column with the median age of the dataset.

In [3]:
df['Age'] = df['Age'].fillna(df['Age'].median()) # inplace=True modifies the original dataframe without needing to assign it back to df
# so we can write df['Age'].fillna(df['Age'].median(), inplace=True) instead of df = df['Age'].fillna(df['Age'].median())
# but i'm not using it because it can be confusing when chaining methods, so I prefer to assign it back to df,  this way we prevent a warning from pandas about the SettingWithCopyWarning

In [4]:
df['Age'] = df['Age'].fillna(df['Age'].median())
print (f'number of missing values in Age column after filling: {df["Age"].isnull().sum()}') # sum of boolean values gives the count of True values, which in this case is the count of missing values

number of missing values in Age column after filling: 0


## Cabin

Since we saw in the EDA that the Cabin column has around 77% missing values, we can safely drop it or we can create a new feature called HasCabin, which will be 1 if the passenger has a cabin and 0 if not. This way we can keep the information about whether the passenger had a cabin or not, without losing too much information from the dataset.

So first we will investigate if the HasCabin feature is correlated with the target variable Survived, and if it is, we will keep it, otherwise we will drop it.

In [5]:
df['HasCabin'] = df['Cabin'].apply(lambda x: 0 if pd.isnull(x) else 1)

print(f'Survival rate by HasCabin:\n{df.groupby("HasCabin")["Survived"].mean()}')

Survival rate by HasCabin:
HasCabin
0    0.299854
1    0.666667
Name: Survived, dtype: float64


Passengers with a cabin had significant higher survival rate. So we'll keep HasCabin featuer and drop the Cabin column.

In [6]:
df = df.drop(columns=["Cabin"])

## Embarked

There is two missing values in the Embarked column, we can fill them with the most common value (mode) of the column.

In [7]:
print('Missing embarked values before filling:', df['Embarked'].isnull().sum())

print(f'Most common embarked value: {df["Embarked"].mode()[0]}')


df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print(f'Missing Embarked values after filling: {df["Embarked"].isnull().sum()}')

Missing embarked values before filling: 2
Most common embarked value: S
Missing Embarked values after filling: 0


## Confirm no missing values remaining

In [8]:
print(df.isnull().sum())  # Check for any remaining missing values in the dataset
print(f'\nTotal missing: {df.isnull().sum().sum()}')  # Total number of missing values in the entire dataset

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64

Total missing: 0


# Feature Engineering

## Family Size

Calculate family size by summing siblings/spouses and parents/children, adding 1 for the individual.

In [9]:
df['FamilySize'] = df["SibSp"] + df["Parch"] + 1 

## IsAlone

 Create a new column IsAlone where 1 indicates the passenger is alone, and 0 otherwise.

In [10]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int) 

print(f'Number of passengers traveling alone: {df["IsAlone"].sum()}')

print(f'Survival rate by IsAlone:\n{df.groupby("IsAlone")["Survived"].mean()}')

Number of passengers traveling alone: 537
Survival rate by IsAlone:
IsAlone
0    0.505650
1    0.303538
Name: Survived, dtype: float64


## Age Categories

In [11]:
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 35, 60, np.inf], labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"])

print(df["AgeGroup"].value_counts().sort_index())
print(f'Survival rate by AgeGroup:\n{df.groupby("AgeGroup")["Survived"].mean()}')

AgeGroup
Child           69
Teenager        70
Young Adult    535
Adult          195
Senior          22
Name: count, dtype: int64
Survival rate by AgeGroup:
AgeGroup
Child          0.579710
Teenager       0.428571
Young Adult    0.353271
Adult          0.400000
Senior         0.227273
Name: Survived, dtype: float64


Child and Teenager have higher survival rates compared to Young Adult, Adult, and Senior groups. 

# Fare Bins

Create fare bins using quantiles to categorize passengers into four groups based on their fare

In [12]:
# Create fare bins using quantiles to categorize passengers into four groups based on their fare
df['FareBin'] = pd.qcut(df['Fare'], 4, labels=['Low', 'Medium', 'High', 'Very High'])

print(f'Survival rate by FareBin:\n{df.groupby("FareBin")["Survived"].mean()}')

Survival rate by FareBin:
FareBin
Low          0.197309
Medium       0.303571
High         0.454955
Very High    0.581081
Name: Survived, dtype: float64


# Encode Categorical Variables

We should encode categorical variables into numerical values for machine learning models.

In [13]:
print(df.dtypes)
print(f'Shape: {df.shape}')

PassengerId       int64
Survived          int64
Pclass            int64
Name                str
Sex                 str
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked            str
HasCabin          int64
FamilySize        int64
IsAlone           int64
AgeGroup       category
FareBin        category
dtype: object
Shape: (891, 16)


In [14]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

df = pd.get_dummies(df, columns=['Embarked', "AgeGroup", "FareBin"], drop_first=True, dtype=int)

In [15]:
print(f'Shape: {df.shape}')
df.head()

Shape: (891, 22)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,IsAlone,Embarked_Q,Embarked_S,AgeGroup_Teenager,AgeGroup_Young Adult,AgeGroup_Adult,AgeGroup_Senior,FareBin_Medium,FareBin_High,FareBin_Very High
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,...,0,0,1,0,1,0,0,0,0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,...,0,0,0,0,0,1,0,0,0,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,...,1,0,1,0,1,0,0,1,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,...,0,0,1,0,1,0,0,0,0,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,...,1,0,1,0,1,0,0,1,0,0


# Drop columns

Drop unnecessary columns that won't be used in the model.

In [16]:
drop_columns = ['PassengerId', 'Name', 'Ticket']
df = df.drop(columns=drop_columns)

print(f'Shape after dropping columns {drop_columns}: {df.shape}')

print('Final columns in the dataset:', df.columns.tolist())

Shape after dropping columns ['PassengerId', 'Name', 'Ticket']: (891, 19)
Final columns in the dataset: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'FamilySize', 'IsAlone', 'Embarked_Q', 'Embarked_S', 'AgeGroup_Teenager', 'AgeGroup_Young Adult', 'AgeGroup_Adult', 'AgeGroup_Senior', 'FareBin_Medium', 'FareBin_High', 'FareBin_Very High']


# Final Check

In [17]:
print(f'Missing values after cleaning: {df.isnull().sum().sum()}')
print(f'Final shape of the dataset: {df.dtypes}')

Missing values after cleaning: 0
Final shape of the dataset: Survived                  int64
Pclass                    int64
Sex                       int64
Age                     float64
SibSp                     int64
Parch                     int64
Fare                    float64
HasCabin                  int64
FamilySize                int64
IsAlone                   int64
Embarked_Q                int64
Embarked_S                int64
AgeGroup_Teenager         int64
AgeGroup_Young Adult      int64
AgeGroup_Adult            int64
AgeGroup_Senior           int64
FareBin_Medium            int64
FareBin_High              int64
FareBin_Very High         int64
dtype: object


All features are numeric (int64, float64).

# Save Final Data

In [18]:
df.to_csv('../data/cleaned_train.csv', index=False)  # Save the cleaned dataset to a new CSV file
print('Cleaned dataset saved to ../data/cleaned_train.csv')
print(f'Final shape of the dataset: {df.shape}')

df.head()

Cleaned dataset saved to ../data/cleaned_train.csv
Final shape of the dataset: (891, 19)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,HasCabin,FamilySize,IsAlone,Embarked_Q,Embarked_S,AgeGroup_Teenager,AgeGroup_Young Adult,AgeGroup_Adult,AgeGroup_Senior,FareBin_Medium,FareBin_High,FareBin_Very High
0,0,3,0,22.0,1,0,7.2500,0,2,0,0,1,0,1,0,0,0,0,0
1,1,1,1,38.0,1,0,71.2833,1,2,0,0,0,0,0,1,0,0,0,1
2,1,3,1,26.0,0,0,7.9250,0,1,1,0,1,0,1,0,0,1,0,0
3,1,1,1,35.0,1,0,53.1000,1,2,0,0,1,0,1,0,0,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,1,1,0,1,0,1,0,0,1,0,0
